# 04_tema — Langkah 13
Jalankan dari atas ke bawah. Sel 1 dan Sel 2 adalah SEL PEMBUKA.

## Sel 1 — SEL PEMBUKA (Python)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%load_ext rpy2.ipython
!apt-get -qq update > /dev/null
!apt-get -qq install -y libpoppler-cpp-dev libglpk-dev libxml2-dev libcurl4-openssl-dev libssl-dev libfontconfig1-dev libharfbuzz-dev libfribidi-dev libfreetype6-dev libpng-dev libtiff5-dev libjpeg-dev > /dev/null
print('Sel 1 selesai')

## Sel 2 — SEL PEMBUKA (R)

In [ ]:
%%R
proj <- "/content/drive/MyDrive/bibliometrik-cxr"
lib  <- file.path(proj, "Rlib")
dir.create(lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(lib, .libPaths()))
codename <- system("lsb_release -cs", intern = TRUE)
options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename)))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
  paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c("bibliometrix", "UpSetR", "readxl", "writexl", "ggplot2", "htmlwidgets")
baru <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(baru)) install.packages(baru, lib = lib)
setwd(proj)
suppressPackageStartupMessages(library(bibliometrix))
cat("Folder kerja:", getwd(), "\nbibliometrix", as.character(packageVersion("bibliometrix")), "siap\n")

## Sel 3 — Cek pengaruh COVID-19 terhadap tren tahunan

In [ ]:
%%R
M <- readRDS("03_clean/M.rds")
txt <- tolower(paste(M$TI, M$AB, M$DE))
covid <- grepl("covid|sars-cov|coronavirus", txt)
tab <- table(Tahun = M$PY, COVID = ifelse(covid, "covid", "non_covid"))
print(tab)
cat("Total dokumen bertema COVID-19:", sum(covid), "dari", nrow(M), "\n")

## Sel 4 — Frekuensi author keywords (cek tema imbalance dan multi-label)

In [ ]:
%%R
M <- readRDS("03_clean/M.rds")
kw <- trimws(unlist(strsplit(M$DE, ";")))
kw <- kw[kw != "" & !is.na(kw)]
cat("Jumlah keyword unik:", length(unique(kw)), "\n\n40 keyword teratas:\n")
print(head(sort(table(kw), decreasing = TRUE), 40))
cat("\nKeyword terkait imbalance, multi-label, long-tail, shift:\n")
print(sort(table(kw[grepl("IMBALANC|MULTI-?LABEL|MULTILABEL|LONG-?TAIL|RARE|MINORITY|SHIFT|CONTINUAL|PARTIAL", kw)]), decreasing = TRUE))

## Sel 5 — Thematic map dan evolution, versi revisi
Istilah pencarian (semi-supervised, chest X-ray) dibuang dari peta karena ada di semua dokumen, sehingga sub-tema terlihat.

In [ ]:
%%R
library(bibliometrix); library(writexl); library(ggplot2)
M <- readRDS("03_clean/M.rds")

sin <- c(
  "PSEUDO-LABELING;PSEUDO LABELING;PSEUDO-LABELLING;PSEUDO LABEL;PSEUDO-LABEL;PSEUDO LABELS;PSEUDO-LABELS",
  "CHEST X-RAY;CHEST X-RAYS;CXR;CHEST X-RAY (CXR);CHEST RADIOGRAPH;CHEST RADIOGRAPHY;CHEST RADIOGRAPHS;CHEST X-RAY IMAGES;CHEST X-RAY IMAGE",
  "SEMI-SUPERVISED LEARNING;SEMI SUPERVISED LEARNING;SEMISUPERVISED LEARNING;SSL;SEMI-SUPERVISED;SEMI-SUPERVISED DEEP LEARNING;SEMI-SUPERVISED CLASSIFICATION",
  "CLASS IMBALANCE;IMBALANCED DATA;DATA IMBALANCE;IMBALANCED LEARNING;IMBALANCED DATASET",
  "MULTI-LABEL CLASSIFICATION;MULTILABEL CLASSIFICATION;MULTI LABEL CLASSIFICATION;MULTI-LABEL",
  "COVID-19;COVID 19;COVID;SARS-COV-2;CORONAVIRUS",
  "MEDICAL IMAGE SEGMENTATION;SEGMENTATION;IMAGE SEGMENTATION",
  "MEDICAL IMAGE CLASSIFICATION;IMAGE CLASSIFICATION;CLASSIFICATION;CHEST X-RAY CLASSIFICATION")
hapus <- c("DEEP LEARNING", "MACHINE LEARNING", "ARTIFICIAL INTELLIGENCE",
           "SEMI-SUPERVISED LEARNING", "CHEST X-RAY", "MEDICAL IMAGING", "MEDICAL IMAGE")

tm <- thematicMap(M, field = "DE", n = 250, minfreq = 3, stemming = FALSE,
                  size = 0.5, n.labels = 3, repel = TRUE,
                  synonyms = sin, remove.terms = hapus)
ggsave("05_figures/thematic_map_v2.png", tm$map, width = 9, height = 7, dpi = 300)
write_xlsx(list(Klaster = tm$clusters, Kata = tm$words), "05_figures/thematic_map_v2.xlsx")
print(tm$clusters[, c("groups", "name_full", "n", "centrality", "density")], width = 200)

te <- thematicEvolution(M, field = "DE", years = c(2020, 2022), n = 250,
                        minFreq = 2, synonyms = sin, remove.terms = hapus)
evo <- plotThematicEvolution(te$Nodes, te$Edges)
htmlwidgets::saveWidget(evo, "05_figures/thematic_evolution.html", selfcontained = FALSE)
cat("Selesai\n")

In [ ]:
%%R
library(writexl)
write_xlsx(list(Nodes = te$Nodes, Edges = te$Edges),
           "05_figures/thematic_evolution_data.xlsx")
cat("Tersimpan:", file.exists("05_figures/thematic_evolution_data.xlsx"), "\n")